In [ ]:
"""
Cross-Project Vulnerability Detection
-------------------------------------
Train on Debian → Test on Chrome
"""

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam

import warnings
warnings.filterwarnings("ignore")


import tensorflow as tf
import random
import os

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)


# =========================================================
# Neural Network
# =========================================================

def create_neural_network(input_dim: int):
    model = Sequential([
        Dense(256, activation='relu', input_dim=input_dim),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================

def semi_supervised_transfer_learning(x_train, y_train, x_test, class_weight_dict):

    model = create_neural_network(x_train.shape[1])

    # Train on source (Debian)
    model.fit(
        x_train, y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        class_weight=class_weight_dict,
        verbose=2
    )

    # Pseudo-label target (Chrome)
    y_pred = model.predict(x_test, verbose=0)
    y_pred_binary = (y_pred > 0.5).astype(int).flatten()

    # Augment source with pseudo-labelled target
    x_train_aug = np.concatenate((x_train, x_test))
    y_train_aug = np.concatenate((y_train, y_pred_binary))

    # Retrain
    model.fit(
        x_train_aug, y_train_aug,
        epochs=30,
        batch_size=32,
        class_weight=class_weight_dict,
        verbose=0
    )

    return model, y_pred.flatten()


# =========================================================
# XGBoost
# =========================================================

def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Main
# =========================================================

def main():

    print("\n=== Training on Debian → Testing on Chrome ===")

    # Load embeddings
    deb_emb = np.load('dataset/debian_embeddings.npy')
    deb_lbl = np.load('dataset/debian_labels.npy')

    chr_emb = np.load('dataset/chrome_embeddings.npy')
    chr_lbl = np.load('dataset/chrome_labels.npy')

    # Normalize
    scaler = StandardScaler()
    deb_emb = scaler.fit_transform(deb_emb)
    chr_emb = scaler.transform(chr_emb)

    # Compute class weights (based on Debian only)
    classes = np.unique(deb_lbl)
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=deb_lbl
    )
    class_weight_dict = {cls: w for cls, w in zip(classes, class_weights)}

    # Ensemble
    num_iterations = 3
    ensemble_predictions = np.zeros(len(chr_lbl))

    for i in range(num_iterations):

        print(f"[Iteration {i+1}/{num_iterations}]")

        model_nn, _ = semi_supervised_transfer_learning(
            deb_emb, deb_lbl,
            chr_emb,
            class_weight_dict
        )

        # Confidence-based weighting
        y_train_pred = model_nn.predict(deb_emb, verbose=0).flatten()
        sample_weights = np.where(
            deb_lbl == 1,
            y_train_pred,
            1 - y_train_pred
        )

        # Train XGBoost
        model_xgb = train_base_model_xgb(
            deb_emb, deb_lbl,
            sample_weights
        )

        # Predict on Chrome
        model_xgb_pred = model_xgb.predict(chr_emb)
        ensemble_predictions += model_xgb_pred

    # Average predictions
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    # Metrics
    accuracy = accuracy_score(chr_lbl, y_pred_final)
    recall = recall_score(chr_lbl, y_pred_final)
    precision = precision_score(chr_lbl, y_pred_final)
    auc = roc_auc_score(chr_lbl, ensemble_avg)
    f1 = f1_score(chr_lbl, y_pred_final)

    tn, fp, fn, tp = confusion_matrix(chr_lbl, y_pred_final).ravel()
    g_mean = np.sqrt(tp / (tp + fn))
    pf = fp / (fp + tn)

    print("\n=== Evaluation Results (Chrome Test Set) ===")
    print(f"Accuracy:   {accuracy:.3f}")
    print(f"Precision:  {precision:.3f}")
    print(f"Recall:     {recall:.3f}")
    print(f"F1-score:   {f1:.3f}")
    print(f"AUC:        {auc:.3f}")
    print(f"G-mean:     {g_mean:.3f}")
    print(f"PF value:   {pf:.3f}")

    y_pred_final = (ensemble_avg > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(chr_lbl, y_pred_final).ravel()
    print("Confusion Matrix:")
    print(f"TN: {tn}")
    print(f"FP: {fp}")
    print(f"FN: {fn}")
    print(f"TP: {tp}")

    # =========================================================
    # Plot Confusion Matrix
    # =========================================================

    cm = confusion_matrix(chr_lbl, y_pred_final)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest')
    plt.title("Confusion Matrix (Chrome Test Set)")
    plt.colorbar()

    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["Non-Vulnerable", "Vulnerable"])
    plt.yticks(tick_marks, ["Non-Vulnerable", "Vulnerable"])

    # Annotate values
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j],
                     ha="center", va="center")

    plt.ylabel("Actual Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()



=== Training on Debian → Testing on Chrome ===
[Iteration 1/3]
Epoch 1/50
